# 07c · The question: a prompt becomes ψ, in C (paper §5)
**Requires the C kernel** (`jupyter-c-kernel`; each cell is a standalone C program compiled with gcc). The functions below are extracted **verbatim** from `VAPMIP/PtolC/ptol.c` at build time:
`P[16]` (lines 170–172), `project` (282–293), `measure_sigma` (311–322), `measure_gamma` (333–341). The last cell runs the real `ptol -r` and compares.

In [1]:
//%cflags:-D_DEFAULT_SOURCE
//%ldflags:-lm
#include <stdio.h>
#include <string.h>
#include <math.h>
#include <stdlib.h>

/* ptol.c:170-172 */
static const int P[16] = {
    2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53
};

/* ptol.c:282-293 */
static double project(const unsigned char *s, int n, int k, double sig)
{
    double sum  = 0.0;
    double freq = 2.0 * M_PI / (double)P[k];
    int j_blue  = (k >= 4 && k <= 7) || (k >= 12 && k <= 15);
    for (int i = 1; i <= n; i++) {
        double phase = freq * (double)i;
        double w     = j_blue ? sin(phase) : cos(phase);
        sum += (double)s[i-1] * pow((double)i, -sig) * w;
    }
    return sum;
}

/* ptol.c:311-322 */
static double measure_sigma(const double *v)
{
    double p_red = 0.0, p_blue = 0.0;
    for (int k = 0; k < 16; k++) {
        int j_blue = (k >= 4 && k <= 7) || (k >= 12 && k <= 15);
        if (j_blue) p_blue += v[k] * v[k];
        else        p_red  += v[k] * v[k];
    }
    double total = p_red + p_blue;
    if (total < 1e-15) return 0.5;
    return p_red / total;
}

/* ptol.c:333-341 */
static void measure_gamma(const double *v, double *gamma_out, double *u_out)
{
    double sig = measure_sigma(v);              /* P_red / (P_red + P_blue) */
    double g   = 2.0 * sig - 1.0;               /* Γ ∈ (−1, 1)              */
    if (g >  0.999999999) g =  0.999999999;
    if (g < -0.999999999) g = -0.999999999;
    if (gamma_out) *gamma_out = g;
    if (u_out)     *u_out     = 2.0 * atanh(g); /* = ln(P_red/P_blue)       */
}

int main(void) {
    const char *prompts[] = {"finger pointer", "what is a tree", "neutron generation in different reactor types",
                              "good morning", "can you play the piano?", "the first reactor was built after neutrons were discovered"};
    for (int q = 0; q < 6; q++) {
        const unsigned char *s = (const unsigned char *)prompts[q]; int n = (int)strlen(prompts[q]);
        double x[16], v[16], norm = 0.0;
        for (int k = 0; k < 16; k++) { x[k] = project(s, n, k, 0.5); norm += x[k] * x[k]; }
        norm = sqrt(norm);
        for (int k = 0; k < 16; k++) v[k] = (norm > 0.0) ? x[k] / norm : 0.0;
        double gam, u; measure_gamma(v, &gam, &u);
        printf("%-58s sigma_self=%.10f  Gamma=%+.10f  u=%+.6f\n", prompts[q], measure_sigma(v), gam, u);
    }
    return 0;
}


finger pointer                                             sigma_self=0.1501236969  Gamma=-0.6997526061  u=-1.733631
what is a tree                                             sigma_self=0.2100110014  Gamma=-0.5799779971  u=-1.324859


neutron generation in different reactor types              sigma_self=0.2453821540  Gamma=-0.5092356919  u=-1.123395
good morning                                               sigma_self=0.2530896120  Gamma=-0.4938207760  u=-1.082202
can you play the piano?                                    sigma_self=0.0307495329  Gamma=-0.9385009342  u=-3.450648
the first reactor was built after neutrons were discovered sigma_self=0.1304088690  Gamma=-0.7391822620  u=-1.897348


## The real binary against the extracted functions (Eye H, σ = ½)

In [2]:
//%cflags:-D_DEFAULT_SOURCE
//%ldflags:-lm
#include <stdio.h>
#include <string.h>
#include <math.h>
#include <stdlib.h>

static const int P[16] = {
    2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53
};

static double project(const unsigned char *s, int n, int k, double sig)
{
    double sum  = 0.0;
    double freq = 2.0 * M_PI / (double)P[k];
    int j_blue  = (k >= 4 && k <= 7) || (k >= 12 && k <= 15);
    for (int i = 1; i <= n; i++) {
        double phase = freq * (double)i;
        double w     = j_blue ? sin(phase) : cos(phase);
        sum += (double)s[i-1] * pow((double)i, -sig) * w;
    }
    return sum;
}

int main(void) {
    const char *prompt = "what is a tree";
    system("mkdir -p /tmp/ptol_nb_home/.ptolemy");
    FILE *p = popen("HOME=/tmp/ptol_nb_home ../../PtolC/ptol -r \"what is a tree\" 2>/dev/null", "r");
    if (!p) { printf("FAIL: could not run ../../PtolC/ptol (build it: cd PtolC && make)\n"); return 1; }
    double real[16]; int got = 0; char line[256];
    while (got < 16 && fgets(line, sizeof line, p)) real[got++] = atof(line);
    pclose(p);
    const unsigned char *s = (const unsigned char *)prompt; int n = (int)strlen(prompt);
    double x[16], norm = 0.0, worst = 0.0;
    for (int k = 0; k < 16; k++) { x[k] = project(s, n, k, 0.5); norm += x[k] * x[k]; }
    norm = sqrt(norm);
    for (int k = 0; k < 16; k++) { double d = fabs(x[k] / norm - real[k]); if (d > worst) worst = d; }
    printf("read %d numbers from ptol -r; max |extracted - real| = %.2e\n", got, worst);
    printf("%s: the extracted project() reproduces ptol -r to the printed precision (1e-10)\n", (got == 16 && worst < 1e-9) ? "PASS" : "FAIL");
    return 0;
}


read 16 numbers from ptol -r; max |extracted - real| = 4.87e-11
PASS: the extracted project() reproduces ptol -r to the printed precision (1e-10)
